In [13]:
import sys

In [14]:
import json
# import spacy
from neo4j import GraphDatabase

In [15]:
from sentence_transformers import CrossEncoder

c:\Users\andre\anaconda3\envs\kag_env1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
import pandas as pd

In [17]:
sys.path.append("../src")

In [18]:
from load_data.funciones_carga_datos import load_filter_dataset_HuggingFace
from conexion_Neo4j.conexion_Neo4j import ConexionNeo4j


In [19]:
database_Neo = "2wiki.prueba.rebel"
conn_Neo4j = ConexionNeo4j(database_Neo)

In [20]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    # database = "2wiki.prueba.rebel.3",
)

# Mejora subgrafo extraido

- Ver direccion de relaciones
- Filtrar saltos desde nodos con muchas relaciones

## Filtrar nodos con muchas relaciones (supernodos) y caminos basura

Filtrar caminos cuyo 2ª nodo tenga mas de N relaciones
Filtrar caminos en los que las 2 relaciones sean igual y las direcciones distintas. 

- Añadida direccion de la relacion
- Añadido grado del nodo (nº de relaciones)

In [ ]:
query_subgrafo = f"""
    MATCH path = (n:Entity)-[*1..2]-(m)
    WHERE n.name IN $entis
    RETURN
        elementId(n) AS ID,
        n.name AS entidad_inicial,
        [node IN nodes(path) | node.name] AS nodos_names,
        [node IN nodes(path) | COUNT{{(node)--()}}] AS degrees,
        [rel IN relationships(path) | type(rel)] AS relaciones_types,
        [i IN range(0, size(relationships(path)) - 1) |
        CASE 
            WHEN startNode(relationships(path)[i]) = nodes(path)[i]
            THEN "OUT"
            ELSE "IN"
        END
        ] AS relaciones_direccion
    """
entidades = ["valentin the good", "the falcon"]
subgrafo_raw, summary, keys = driver.execute_query(
    query_subgrafo,
    entis = entidades,
    k = 2,
    database_ = "2wiki.prueba.rebel.3"
    )


### Con records

In [10]:
subgrafo_raw

[<Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film'] degrees=[3, 37] relaciones_types=['genre'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film', 'a day will come'] degrees=[3, 37, 9] relaciones_types=['genre', 'genre'] relaciones_direccion=['OUT', 'IN']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film', 'just once a great lady'] degrees=[3, 37, 15] relaciones_types=['genre', 'genre'] relaciones_direccion=['OUT', 'IN']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film', 'glamour boy'] degrees=[3, 37, 12] relaciones_types=['genre', 'genre'] relaciones_direccion=['OUT', 'IN']>,
 <Record ID='4:8c748

FILTRADO DE RELACIONES CON 2 SALTOS, CUYO 2º NODO TIENE MÁS DE N RELACIONES

In [55]:
subgrafo_raw_filt = [rel for rel in subgrafo_raw if rel['degrees'][1]<10 or len(rel['degrees']) == 2]

In [56]:
subgrafo_raw_filt

[<Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film'] degrees=[3, 37] relaciones_types=['genre'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'martin frič'] degrees=[3, 9] relaciones_types=['director'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'martin frič', 'czech'] degrees=[3, 9, 4] relaciones_types=['director', 'country_of_citizenship'] relaciones_direccion=['OUT', 'OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'martin frič', 'czechoslovakia'] degrees=[3, 9, 1] relaciones_types=['director', 'country_of_citizenship'] relaciones_direccion=['OUT', 'OUT']>,
 <Record ID='4:8c748595-91e4-48

FILTRADO DE CAMINOS DONDE LAS 2 RELACIONES SON IGUALES. Y ADEMAS 1ª OUT y 2ª IN 

In [76]:
subgrafo_raw_filt_2 = [rel for rel in subgrafo_raw_filt if len(rel['relaciones_types'])==1] + [rel for rel in subgrafo_raw_filt if len(rel['relaciones_types'])==2 and (rel['relaciones_types'][0] != rel['relaciones_types'][1] or rel['relaciones_direccion'][0] == rel['relaciones_direccion'][1])]

In [77]:
subgrafo_raw_filt_2

[<Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film'] degrees=[3, 37] relaciones_types=['genre'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'martin frič'] degrees=[3, 9] relaciones_types=['director'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', '1942'] degrees=[3, 9] relaciones_types=['publication_date'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:1547' entidad_inicial='the falcon' nodos_names=['the falcon', 'mystery film'] degrees=[5, 9] relaciones_types=['genre'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:1547' entidad_inicial='the falcon' nodos_names=['the falcon', 'adventure film'] degrees=[5, 

### SUBGRAFO A PANDAS

In [11]:
df_paths = pd.DataFrame(columns=["entidad_incial", "nodo_1", "nodo_2", "nodo_3", "rel_1", "rel_2", "direccion_1", "direccion_2", "grado_1", "grado_2", "grado_3"])

In [11]:
na_row = {
    "entidad_incial": pd.NA,
    "nodo_1": pd.NA,
    "nodo_2": pd.NA,
    "nodo_3": pd.NA,
    "rel_1": pd.NA,
    "rel_2": pd.NA,
    "direccion_1": pd.NA,
    "direccion_2": pd.NA,
    "grado_1": pd.NA,
    "grado_2": pd.NA,
    "grado_3": pd.NA
    }

In [12]:
rels_pandas = []
for path in subgrafo_raw:
    # print(path)
    row = na_row.copy()
    row["entidad_incial"] = path['entidad_inicial']
    row["nodo_1"] = path['nodos_names'][0]
    row["grado_1"] = path['degrees'][0]
    
    for i in range(1, len(path['nodos_names']) ):
        
        row[f"nodo_{i+1}"] = path['nodos_names'][i]
        row[f"grado_{i+1}"] = path['degrees'][i]
        row[f"rel_{i}"] = path['relaciones_types'][i-1]
        row[f"direccion_{i}"] = path['relaciones_direccion'][i-1]
    # print(row)
    rels_pandas.append(row)

In [13]:
df_paths = pd.DataFrame(rels_pandas)

In [14]:
df_paths

,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3
0,valentin the good,valentin the good,comedy film,<NA>,genre,<NA>,OUT,<NA>,3,37,<NA>
1,valentin the good,valentin the good,comedy film,a day will come,genre,genre,OUT,IN,3,37,9
2,valentin the good,valentin the good,comedy film,just once a great lady,genre,genre,OUT,IN,3,37,15
3,valentin the good,valentin the good,comedy film,glamour boy,genre,genre,OUT,IN,3,37,12
4,valentin the good,valentin the good,comedy film,flirting with fate,genre,genre,OUT,IN,3,37,22
...,...,...,...,...,...,...,...,...,...,...,...
86,the falcon,the falcon,1981,sword stained with royal blood,publication_date,publication_date,OUT,IN,5,10,9
87,the falcon,the falcon,1981,banović strahinja,publication_date,publication_date,OUT,IN,5,10,4
88,the falcon,the falcon,1981,12th moscow international film festival,publication_date,point_in_time,OUT,IN,5,10,1
89,the falcon,the falcon,1981,38th venice international film festival,publication_date,point_in_time,OUT,IN,5,10,1


FILTRADO

FILTRADO DE RELACIONES CON 2 SALTOS, CUYO 2º NODO TIENE MÁS DE N RELACIONES

In [15]:
df_paths_filt = df_paths.loc[(df_paths['grado_2'] < 10) | (df_paths['rel_2'].isna())]

In [16]:
df_paths_filt

,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3
0,valentin the good,valentin the good,comedy film,<NA>,genre,<NA>,OUT,<NA>,3,37,<NA>
37,valentin the good,valentin the good,martin frič,<NA>,director,<NA>,OUT,<NA>,3,9,<NA>
38,valentin the good,valentin the good,martin frič,czech,director,country_of_citizenship,OUT,OUT,3,9,4
39,valentin the good,valentin the good,martin frič,czechoslovakia,director,country_of_citizenship,OUT,OUT,3,9,1
40,valentin the good,valentin the good,martin frič,leave it to me,director,director,OUT,IN,3,9,6
41,valentin the good,valentin the good,martin frič,26 august 1968,director,date_of_death,OUT,OUT,3,9,1
42,valentin the good,valentin the good,martin frič,film director,director,occupation,OUT,OUT,3,9,45
43,valentin the good,valentin the good,martin frič,screenwriter,director,occupation,OUT,OUT,3,9,17
44,valentin the good,valentin the good,martin frič,29 march 1902,director,date_of_birth,OUT,OUT,3,9,1
45,valentin the good,valentin the good,martin frič,1955,director,work_period_(start),OUT,OUT,3,9,5


FILTRADO DE CAMINOS DONDE LAS 2 RELACIONES SON IGUALES. Y ADEMAS 1ª OUT y 2ª IN 

In [17]:
df_paths_filt_2 = df_paths_filt.loc[~((df_paths_filt['rel_1'] == df_paths_filt['rel_2']) & (df_paths_filt['direccion_1'] =='OUT') & (df_paths_filt['direccion_2'] =='IN'))]

In [18]:
df_paths_filt_2 = df_paths_filt_2.reset_index(drop = True)

## Tripletas a lenguaje natural

Pasar tripletas a lenguaje natural para el reranker y para el LLM

### CON REGLA MUY BASICA *con records

In [78]:
subgrafo_raw_filt_2[:5]

[<Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'comedy film'] degrees=[3, 37] relaciones_types=['genre'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', 'martin frič'] degrees=[3, 9] relaciones_types=['director'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:560' entidad_inicial='valentin the good' nodos_names=['valentin the good', '1942'] degrees=[3, 9] relaciones_types=['publication_date'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:1547' entidad_inicial='the falcon' nodos_names=['the falcon', 'mystery film'] degrees=[5, 9] relaciones_types=['genre'] relaciones_direccion=['OUT']>,
 <Record ID='4:8c748595-91e4-4824-af8e-5b689e107513:1547' entidad_inicial='the falcon' nodos_names=['the falcon', 'adventure film'] degrees=[5, 

In [ ]:
plantilla_rel = "{entidad_IN} which {rel} is {entidad_OUT}."
caminos_formateados = []
for path in subgrafo_raw_filt_2:
    camino_traducido = []
    for i, rel in enumerate(path['relaciones_types']):
        entidad_origen = path['nodos_names'][0]
        tripleta = {}
        tripleta['rel'] = rel.replace("_", " ")
        ent_1 = path['nodos_names'][i]
        ent_2 = path['nodos_names'][i+1]
        if path['relaciones_direccion'][i] == "OUT":
            tripleta['entidad_IN'] = ent_1
            tripleta['entidad_OUT'] = ent_2
        else:
            tripleta['entidad_IN'] = ent_2
            tripleta['entidad_OUT'] = ent_1
        
        tripleta_natural = plantilla_rel.format(**tripleta)
        camino_traducido.append(tripleta_natural)
        # print(tripleta_natural)
    print(" ".join(camino_traducido))
    caminos_formateados.append(" ".join(camino_traducido))

valentin the good wich genre is comedy film.
valentin the good wich director is martin frič.
valentin the good wich publication date is 1942.
the falcon wich genre is mystery film.
the falcon wich genre is adventure film.
the falcon wich director is vatroslav mimica.
the falcon wich performer is tom conway.
the falcon wich publication date is 1981.
valentin the good wich director is martin frič. martin frič wich country of citizenship is czech.
valentin the good wich director is martin frič. martin frič wich country of citizenship is czechoslovakia.
valentin the good wich director is martin frič. martin frič wich date of death is 26 august 1968.
valentin the good wich director is martin frič. martin frič wich occupation is film director.
valentin the good wich director is martin frič. martin frič wich occupation is screenwriter.
valentin the good wich director is martin frič. martin frič wich date of birth is 29 march 1902.
valentin the good wich director is martin frič. martin frič wi

In [108]:
caminos_formateados

['valentin the good wich genre is comedy film.',
 'valentin the good wich director is martin frič.',
 'valentin the good wich publication date is 1942.',
 'the falcon wich genre is mystery film.',
 'the falcon wich genre is adventure film.',
 'the falcon wich director is vatroslav mimica.',
 'the falcon wich performer is tom conway.',
 'the falcon wich publication date is 1981.',
 'valentin the good wich director is martin frič. martin frič wich country of citizenship is czech.',
 'valentin the good wich director is martin frič. martin frič wich country of citizenship is czechoslovakia.',
 'valentin the good wich director is martin frič. martin frič wich date of death is 26 august 1968.',
 'valentin the good wich director is martin frič. martin frič wich occupation is film director.',
 'valentin the good wich director is martin frič. martin frič wich occupation is screenwriter.',
 'valentin the good wich director is martin frič. martin frič wich date of birth is 29 march 1902.',
 'vale

PRUEBA RERANKING

In [ ]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)
# reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
# reranker = CrossEncoder("jinaai/jina-reranker-v3", max_length=512)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2822.31it/s]
Qwen3ForSequenceClassification LOAD REPORT from: jinaai/jina-reranker-v3
Key                     | Status     | 
------------------------+------------+-
projector.{0, 2}.weight | UNEXPECTED | 
score.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [97]:
question = "Do both films The Falcon (Film) and Valentin The Good have the directors from the same country?"

In [ ]:
pairs = [[question, frase] for frase in caminos_formateados ]
scores_rerank = reranker.predict(pairs)

In [116]:
tripletas_reranked={}
for idx, elem in enumerate(caminos_formateados):
    tripletas_reranked[elem] = scores_rerank[idx].item()

In [117]:
tripletas_reranked

{'valentin the good wich genre is comedy film.': 0.8494153618812561,
 'valentin the good wich director is martin frič.': 0.9157251119613647,
 'valentin the good wich publication date is 1942.': 0.10207981616258621,
 'the falcon wich genre is mystery film.': 0.018540970981121063,
 'the falcon wich genre is adventure film.': 0.02789125218987465,
 'the falcon wich director is vatroslav mimica.': 0.011214667931199074,
 'the falcon wich performer is tom conway.': 4.145564162172377e-05,
 'the falcon wich publication date is 1981.': 0.0005111029604449868,
 'valentin the good wich director is martin frič. martin frič wich country of citizenship is czech.': 0.9750081300735474,
 'valentin the good wich director is martin frič. martin frič wich country of citizenship is czechoslovakia.': 0.9667638540267944,
 'valentin the good wich director is martin frič. martin frič wich date of death is 26 august 1968.': 0.9566138386726379,
 'valentin the good wich director is martin frič. martin frič wich occ

In [ ]:
tripletas_reranked = reranking_tripletas(question, caminos_formateados, reranker)

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

In [122]:
results = model.rerank(question, caminos_formateados)

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

Score: 0.4185
Document: valentin the good wich director is martin frič. martin frič wich country of citizenship is czech....

Score: 0.3922
Document: valentin the good wich director is martin frič. martin frič wich country of citizenship is czechoslo...

Score: 0.3488
Document: valentin the good wich director is martin frič....

Score: 0.2126
Document: the falcon wich director is vatroslav mimica....

Score: 0.1954
Document: valentin the good wich director is martin frič. martin frič wich occupation is film director....

Score: 0.1191
Document: valentin the good wich genre is comedy film....

Score: 0.0846
Document: valentin the good wich director is martin frič. martin frič wich date of death is 26 august 1968....

Score: 0.0677
Document: the falcon wich director is vatroslav mimica. vatroslav mimica wich date of birth is 25 june 1923....

Score: 0.0596
Document: the falcon wich director is vatroslav mimica. vatroslav mimica wich place of birth is omiš....

Score: 0.0519
Document: val

### CON REGLA MUY BASICA  Con pandas

In [19]:
df_paths_filt_2.head()

,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3
0,valentin the good,valentin the good,comedy film,<NA>,genre,<NA>,OUT,<NA>,3,37,<NA>
1,valentin the good,valentin the good,martin frič,<NA>,director,<NA>,OUT,<NA>,3,9,<NA>
2,valentin the good,valentin the good,martin frič,czech,director,country_of_citizenship,OUT,OUT,3,9,4
3,valentin the good,valentin the good,martin frič,czechoslovakia,director,country_of_citizenship,OUT,OUT,3,9,1
4,valentin the good,valentin the good,martin frič,26 august 1968,director,date_of_death,OUT,OUT,3,9,1


In [20]:
df_paths_filt_2 = df_paths_filt_2.fillna("")

In [ ]:
def construir_string(fila):
    if fila['direccion_1'] == 'OUT':
        inicio = 'nodo_1'
        final = 'nodo_2'
    elif fila['direccion_1'] == 'IN':
        inicio = 'nodo_2'
        final = 'nodo_1'
    tripleta_formateada = f"{fila[inicio]} which {fila['rel_1'].replace('_', ' ')} is {fila[final]}."
    if fila['rel_2'] != "":
        if fila['direccion_2'] == 'OUT':
            inicio = 'nodo_2'
            final = 'nodo_3'
        elif fila['direccion_2'] == 'IN':
            inicio = 'nodo_3'
            final = 'nodo_2'
        tripleta_formateada += f" {fila[inicio]} which {fila['rel_2'].replace('_', ' ')} is {fila[final]}"
    # Construimos y retornamos el string completo
    return tripleta_formateada

# Aplicamos la función a todo el DataFrame
df_paths_filt_2['tripleta_formateada'] = df_paths_filt_2.apply(construir_string, axis=1)


In [43]:
df_paths_filt_2

,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3,tripleta_formateada,rerank_score
0,valentin the good,valentin the good,comedy film,,genre,,OUT,,3,37,,valentin the good wich genre is comedy film.,0.982649
1,valentin the good,valentin the good,martin frič,,director,,OUT,,3,9,,valentin the good wich director is martin frič.,0.989693
2,valentin the good,valentin the good,martin frič,czech,director,country_of_citizenship,OUT,OUT,3,9,4,valentin the good wich director is martin frič...,0.998901
3,valentin the good,valentin the good,martin frič,czechoslovakia,director,country_of_citizenship,OUT,OUT,3,9,1,valentin the good wich director is martin frič...,0.998596
4,valentin the good,valentin the good,martin frič,26 august 1968,director,date_of_death,OUT,OUT,3,9,1,valentin the good wich director is martin frič...,0.955195
5,valentin the good,valentin the good,martin frič,film director,director,occupation,OUT,OUT,3,9,45,valentin the good wich director is martin frič...,0.999386
6,valentin the good,valentin the good,martin frič,screenwriter,director,occupation,OUT,OUT,3,9,17,valentin the good wich director is martin frič...,0.999063
7,valentin the good,valentin the good,martin frič,29 march 1902,director,date_of_birth,OUT,OUT,3,9,1,valentin the good wich director is martin frič...,0.880851
8,valentin the good,valentin the good,martin frič,1955,director,work_period_(start),OUT,OUT,3,9,5,valentin the good wich director is martin frič...,0.997025
9,valentin the good,valentin the good,1942,,publication_date,,OUT,,3,9,,valentin the good wich publication date is 1942.,0.130055


In [44]:
df_paths_filt_2['tripleta_formateada'][2]

'valentin the good wich director is martin frič. martin frič wich country of citizenship is czech'

## Reranking mejorado

In [24]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=512)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3468.41it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
question = "Do both films The Falcon (Film) and Valentin The Good have the directors from the same country?"

In [45]:
pairs = [[question, frase] for frase in list(df_paths_filt_2['tripleta_formateada']) ]
scores_rerank = reranker.predict(pairs)

In [47]:
df_paths_filt_2["rerank_score"] = scores_rerank.tolist()

In [48]:
df_paths_filt_2

,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3,tripleta_formateada,rerank_score
0,valentin the good,valentin the good,comedy film,,genre,,OUT,,3,37,,valentin the good wich genre is comedy film.,0.982649
1,valentin the good,valentin the good,martin frič,,director,,OUT,,3,9,,valentin the good wich director is martin frič.,0.989693
2,valentin the good,valentin the good,martin frič,czech,director,country_of_citizenship,OUT,OUT,3,9,4,valentin the good wich director is martin frič...,0.999036
3,valentin the good,valentin the good,martin frič,czechoslovakia,director,country_of_citizenship,OUT,OUT,3,9,1,valentin the good wich director is martin frič...,0.998769
4,valentin the good,valentin the good,martin frič,26 august 1968,director,date_of_death,OUT,OUT,3,9,1,valentin the good wich director is martin frič...,0.973016
5,valentin the good,valentin the good,martin frič,film director,director,occupation,OUT,OUT,3,9,45,valentin the good wich director is martin frič...,0.999386
6,valentin the good,valentin the good,martin frič,screenwriter,director,occupation,OUT,OUT,3,9,17,valentin the good wich director is martin frič...,0.999063
7,valentin the good,valentin the good,martin frič,29 march 1902,director,date_of_birth,OUT,OUT,3,9,1,valentin the good wich director is martin frič...,0.928322
8,valentin the good,valentin the good,martin frič,1955,director,work_period_(start),OUT,OUT,3,9,5,valentin the good wich director is martin frič...,0.997374
9,valentin the good,valentin the good,1942,,publication_date,,OUT,,3,9,,valentin the good wich publication date is 1942.,0.329527


In [49]:
df_paths_filt_2['entidad_incial'].unique()

array(['valentin the good', 'the falcon'], dtype=object)

In [55]:
# Reemplaza 'A' por tu columna de keys y 'B' por la columna de valores
df_maximos = df_paths_filt_2.groupby('entidad_incial').apply(lambda x: x.nlargest(3, 'rerank_score')).reset_index(drop=True)


C:\Users\andre\AppData\Local\Temp\ipykernel_20676\3223863152.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_maximos = df_paths_filt_2.groupby('entidad_incial').apply(lambda x: x.nlargest(3, 'rerank_score')).reset_index(drop=True)


In [56]:
df_maximos

,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3,tripleta_formateada,rerank_score
0,the falcon,the falcon,adventure film,,genre,,OUT,,5,3,,the falcon wich genre is adventure film.,0.464702
1,the falcon,the falcon,vatroslav mimica,omiš,director,place_of_birth,OUT,OUT,5,5,1,the falcon wich director is vatroslav mimica. ...,0.223124
2,the falcon,the falcon,mystery film,,genre,,OUT,,5,9,,the falcon wich genre is mystery film.,0.168486
3,valentin the good,valentin the good,martin frič,film director,director,occupation,OUT,OUT,3,9,45,valentin the good wich director is martin frič...,0.999386
4,valentin the good,valentin the good,martin frič,screenwriter,director,occupation,OUT,OUT,3,9,17,valentin the good wich director is martin frič...,0.999063
5,valentin the good,valentin the good,martin frič,czech,director,country_of_citizenship,OUT,OUT,3,9,4,valentin the good wich director is martin frič...,0.999036


In [54]:
df_maximos['tripleta_formateada'][1]

'the falcon wich director is vatroslav mimica. vatroslav mimica wich place of birth is omiš'

## Funciones para implementar

CAMBIOS: 
-   Subgrafo ahora devuelve tambien direccion de las relaciones y grado de los nodos (nº de relaciones)
-   Se hace todo el trabajo en pandas. Subgrafo se pasa directo a df y de ahi se saca lo demas.
-   Filtrado del subgrafo inicial, se quitan caminos que :
-       con 2 saltos, cuyo 2º nodo tiene mas de N relaciones
-        donde las 2 relaciones son iguales y 1ª OUT y 2ª IN
-   Tripletas (caminos) a lenhuaje natiural. De momento sencilo - Solo se allade which/is
-   Reranker extendido, ahora:
-       Reranking de solo las relaciones
-       Escalado del reranking de las relaciones (xq salen numeros muy bajos)
-       Ponderado de reranking de la tripleta + reranking de la relacion

Nueva funcion extraer subgrafo.

In [5]:
# def extraer_subgrafo_completo_nueva(self, entidades, n_saltos):
def extraer_subgrafo_completo_nueva(entidades, n_saltos):
    query_subgrafo = f"""
        MATCH path = (n:Entity)-[*1..{n_saltos}]-(m)
        WHERE n.name IN $entis
        RETURN
            elementId(n) AS ID,
            n.name AS entidad_inicial,
            [node IN nodes(path) | node.name] AS nodos_names,
            [node IN nodes(path) | COUNT{{(node)--()}}] AS degrees,
            [rel IN relationships(path) | type(rel)] AS relaciones_types,
            [i IN range(0, size(relationships(path)) - 1) |
            CASE 
                WHEN startNode(relationships(path)[i]) = nodes(path)[i]
                THEN "OUT"
                ELSE "IN"
            END
            ] AS relaciones_direccion
        """
    subgrafo_raw, summary, keys = driver.execute_query(
        query_subgrafo,
        entis = entidades,
        k = 2,
        database_ = "2wiki.prueba.rebel.3"
        )
    return subgrafo_raw

Subgrafo a Pandas

In [ ]:
def subgrafo_a_pandas(subgrafo):
    na_row = {
        "entidad_incial": "",
        "nodo_1": "",
        "nodo_2": "",
        "nodo_3": "",
        "rel_1": "",
        "rel_2": "",
        "direccion_1": "",
        "direccion_2": "",
        "grado_1": "",
        "grado_2": "",
        "grado_3": ""
        }
    rels_pandas = []
    for path in subgrafo:
        row = na_row.copy()
        row["entidad_incial"] = path['entidad_inicial']
        row["nodo_1"] = path['nodos_names'][0]
        row["grado_1"] = path['degrees'][0]
        for i in range(1, len(path['nodos_names']) ):
            row[f"nodo_{i+1}"] = path['nodos_names'][i]
            row[f"grado_{i+1}"] = path['degrees'][i]
            row[f"rel_{i}"] = path['relaciones_types'][i-1]
            row[f"direccion_{i}"] = path['relaciones_direccion'][i-1]
        rels_pandas.append(row)
    df_subgrafo = pd.DataFrame(rels_pandas)
    return df_subgrafo

Filtrado basura subgrafo

In [ ]:
def filtrado_subgrafo(df_subgrafo, n_rel_max):
    """
    Filtra:
        - Caminos con 2 saltos, cuyo 2º nodo tiene mas de N relaciones
        - Caminos donde las 2 relaciones son iguales y 1ª OUT y 2ª IN
    """
    # df_subgrafo_filt = df_subgrafo.loc[(df_subgrafo['grado_2'] < n_rel_max) | (df_subgrafo['rel_2'].isna())]
    df_subgrafo_filt = df_subgrafo.loc[(df_subgrafo['grado_2'] < n_rel_max) | (df_subgrafo['rel_2'] == "")]
    df_subgrafo_filt = df_subgrafo_filt.loc[~((df_subgrafo_filt['rel_1'] == df_subgrafo_filt['rel_2']) & (df_subgrafo_filt['direccion_1'] =='OUT') & (df_subgrafo_filt['direccion_2'] =='IN'))].reset_index(drop=True)
    return df_subgrafo_filt
    

Formateo de tripletas a lenguaje natural.

In [8]:
def construir_string(fila):
    if fila['direccion_1'] == 'OUT':
        inicio = 'nodo_1'
        final = 'nodo_2'
    elif fila['direccion_1'] == 'IN':
        inicio = 'nodo_2'
        final = 'nodo_1'
    tripleta_formateada = f"{fila[inicio]} which {fila['rel_1'].replace('_', ' ')} is {fila[final]}."
    if fila['rel_2'] != "":
        if fila['direccion_2'] == 'OUT':
            inicio = 'nodo_2'
            final = 'nodo_3'
        elif fila['direccion_2'] == 'IN':
            inicio = 'nodo_3'
            final = 'nodo_2'
        tripleta_formateada += f" {fila[inicio]} which {fila['rel_2'].replace('_', ' ')} is {fila[final]}"
    # Construimos y retornamos el string completo
    return tripleta_formateada

reranker

In [9]:
def reranking_tripletas_pandas(question, df_subgrafo, reranker):
    pairs = [[question, frase] for frase in list(df_subgrafo['tripleta_formateada']) ]
    scores_rerank = reranker.predict(pairs)
    df_subgrafo["rerank_score"] = scores_rerank.tolist()
    return df_subgrafo

In [40]:
def reranking_relaciones_pandas(question, df_subgrafo, reranker):
    rels_1 = list(df_subgrafo['rel_1'])
    rels_2 = list(df_subgrafo['rel_2'])
    rels_to_rerank = []
    for i, rel in enumerate(rels_1):
        rel_to_rerank = rel.replace("_", " ")
        if rels_2[i] != "":
            rel_to_rerank += f" -> {rels_2[i].replace('_', ' ')}"
        rels_to_rerank.append(rel_to_rerank)
    pairs = [[question, relacion] for relacion in rels_to_rerank ]
    scores_rerank = reranker.predict(pairs)
    df_subgrafo["rerank_score_rels"] = scores_rerank.tolist()
    return df_subgrafo, rels_to_rerank

In [158]:
import numpy as np

In [163]:
def escalado_rerank_rels(df_subgrafo):
    # df_subgrafo = df_subgrafo.reset_index(drop=True)
    # df_subgrafo["rerank_score_rels"] = df_subgrafo["rerank_score_rels"].round(6)
    # df_subgrafo["log_score"] = np.log(df_subgrafo["rerank_score_rels"])
    # max_score = df_subgrafo["rerank_score_rels"].max().item()
    # min_score = df_subgrafo["rerank_score_rels"].min().item()
    # max_score = df_subgrafo["log_score"].max().item()
    # min_score = df_subgrafo["log_score"].min().item()
    # df_subgrafo["rerank_score_rels_escalado"] = (df_subgrafo["rerank_score_rels"] - min_score) / (max_score - min_score)
    # df_subgrafo["rerank_score_rels_escalado"] = (df_subgrafo["log_score"] - min_score) / (max_score - min_score)
    
    # Con rank poner valores entre 0 y 1 pero distribuidos normalmente para ue no salgan valores muy pequeños

    min_rank = df_subgrafo["rerank_score_rels"].rank().min()
    max_rank = df_subgrafo["rerank_score_rels"].rank().max()


    df_subgrafo["rerank_score_rels_escalado"] = (df_subgrafo["rerank_score_rels"].rank() - min_rank) / (max_rank - min_rank)
    
    
    return df_subgrafo

In [111]:
def ponderacion_score_reranking_rels(df_subgrafo, peso_tripleta, peso_rel):
    df_subgrafo["score_rerank_ponderado"] = df_subgrafo["rerank_score"] * peso_tripleta + df_subgrafo["rerank_score_rels"] * peso_rel
    return df_subgrafo

In [196]:
def filtrar_por_reranker_pandas(df_subgrafo, n_maximos, min_score, col_score):
    """
    Filtrar los n_maximos de cada entidad
    """
    df_subgrafo = df_subgrafo[df_subgrafo[col_score] >= min_score]
    df_maximos = df_subgrafo.groupby('entidad_incial').apply(lambda x: x.nlargest(n_maximos, 'rerank_score'), include_groups=False).reset_index(drop=True)
    # df_maximos = df_maximos[df_maximos['rerank_score'] >= min_score]
    
    return df_maximos

PRUEBA

In [96]:
dataset_2Wiki = load_filter_dataset_HuggingFace("xanhho/2wikimultihopqa", 50, "train")

In [97]:
dataset_2Wiki[8]

{'_id': 'b9204363096611ebbdafac1f6bf848b6',
 'type': 'comparison',
 'question': 'Which film came out first, The Love Route or Engal Aasan?',
 'context': '[["Engal Aasan", ["Engal Aasan is a 2009 Tamil action comedy- drama film directed by R. K. Kalaimani.", "The film starring Vijayakanth in the lead role and Vikranth, Sheryl Brindo, Akshaya and Suja Varunee playing supporting roles.", "It began its first schedule on 12 March 2008 and released in July 2009.", "The film, upon release could not release the big theatres and became a colossal flop.", "It was dubbed in Telugu as\\" Captain\\"."]], ["Kill the Love", ["Kill the Love is a 1996 South Korean crime drama film."]], ["Where\'s the Love (disambiguation)", ["\\" Where\'s the Love\\" is a 1997 song by Hanson.", "Where\'s the Love may also refer to:\\" Where\'s the Love\\" should not be confused with:"]], ["Route", ["Route or routes may refer to:"]], ["The Love Specialist", ["The Love Specialist is an Italian- French movie filmed in 195

In [98]:
question = dataset_2Wiki[8]['question']

In [99]:
entidades= [
            "first nations",
            "first part",
            "engal aasan",
            "first hill",
            "the love route",
            "romantic drama film"
        ]

In [100]:
driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "password"),
    # database = "2wiki.prueba.rebel.3",
)

In [122]:
subgrafo_raw = extraer_subgrafo_completo_nueva(entidades, 2)

In [102]:
len(subgrafo_raw)

99

In [182]:
subgrafo_df = subgrgafo_a_pandas(subgrafo_raw)

In [183]:
subgrafo_df_filt = filtrado_subgrafo(subgrafo_df, 10)

In [184]:
subgrafo_df_filt['tripleta_formateada'] = subgrafo_df_filt.apply(construir_string, axis=1)

In [106]:
reranker = CrossEncoder("BAAI/bge-reranker-base", max_length=256)
# reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=256)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3776.10it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [185]:
subgrafo_df_reranked = reranking_tripletas_pandas(question, subgrafo_df_filt, reranker)

In [186]:
subgrafo_df_reranked, rels_to_rerank = reranking_relaciones_pandas(question, subgrafo_df_reranked, reranker)
# subgrafo_df_reranked_con_rels, rels_to_rerank = reranking_relaciones_pandas(question, subgrafo_df, reranker)

In [187]:
subgrafo_df_reranked = escalado_rerank_rels(subgrafo_df_reranked)

In [189]:
peso_tripleta = 0.7
peso_rel = 0.3
subgrafo_df_reranked = ponderacion_score_reranking_rels(subgrafo_df_reranked, peso_tripleta, peso_rel)

In [190]:
question

'Which film came out first, The Love Route or Engal Aasan?'

In [191]:
subgrafo_df_reranked


,entidad_incial,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3,tripleta_formateada,rerank_score,rerank_score_rels,rerank_score_rels_escalado,score_rerank_ponderado
0,engal aasan,engal aasan,suja varunee,,cast_member,,OUT,,6,1,,engal aasan which cast member is suja varunee.,0.818501,0.000037,0.222222,0.572962
1,engal aasan,engal aasan,sheryl brindo,,cast_member,,OUT,,6,1,,engal aasan which cast member is sheryl brindo.,0.926737,0.000037,0.222222,0.648727
2,engal aasan,engal aasan,akshaya,,cast_member,,OUT,,6,1,,engal aasan which cast member is akshaya.,0.756974,0.000037,0.222222,0.529893
3,engal aasan,engal aasan,vijayakanth,,cast_member,,OUT,,6,1,,engal aasan which cast member is vijayakanth.,0.684104,0.000037,0.222222,0.478884
4,engal aasan,engal aasan,vikranth,,cast_member,,OUT,,6,1,,engal aasan which cast member is vikranth.,0.773584,0.000037,0.222222,0.541520
5,engal aasan,engal aasan,2009,,publication_date,,OUT,,6,5,,engal aasan which publication date is 2009.,0.824738,0.000037,0.492063,0.577328
6,engal aasan,engal aasan,2009,the international school of columbus,publication_date,inception,OUT,IN,6,5,1,engal aasan which publication date is 2009. th...,0.872334,0.000037,0.428571,0.610645
7,engal aasan,engal aasan,2009,international school of columbus,publication_date,inception,OUT,IN,6,5,3,engal aasan which publication date is 2009. in...,0.815783,0.000037,0.428571,0.571059
8,first hill,first hill,neighborhood,,instance_of,,OUT,,3,1,,first hill which instance of is neighborhood.,0.000037,0.000038,0.666667,0.000037
9,first hill,first hill,seattle,,located_in_the_administrative_territorial_entity,,OUT,,3,6,,first hill which located in the administrative...,0.000037,0.000037,0.587302,0.000037


In [198]:
col_score = "score_rerank_ponderado"
n_maximos = 3
min_score = 0.1

subgrafo_df_reranked_filt = filtrar_por_reranker_pandas(subgrafo_df_reranked, n_maximos, min_score, col_score)

In [199]:
subgrafo_df_reranked_filt

,nodo_1,nodo_2,nodo_3,rel_1,rel_2,direccion_1,direccion_2,grado_1,grado_2,grado_3,tripleta_formateada,rerank_score,rerank_score_rels,rerank_score_rels_escalado,score_rerank_ponderado
0,engal aasan,sheryl brindo,,cast_member,,OUT,,6,1,,engal aasan which cast member is sheryl brindo.,0.926737,0.000037,0.222222,0.648727
1,engal aasan,2009,the international school of columbus,publication_date,inception,OUT,IN,6,5,1,engal aasan which publication date is 2009. th...,0.872334,0.000037,0.428571,0.610645
2,engal aasan,2009,,publication_date,,OUT,,6,5,,engal aasan which publication date is 2009.,0.824738,0.000037,0.492063,0.577328
3,the love route,edward henry peple,,screenwriter,,OUT,,5,1,,the love route which screenwriter is edward he...,0.889109,0.000038,0.698413,0.622388
4,the love route,allan dwan,,director,,OUT,,5,1,,the love route which director is allan dwan.,0.879331,0.000038,0.761905,0.615543
5,the love route,1915,,publication_date,,OUT,,5,11,,the love route which publication date is 1915.,0.860123,0.000037,0.492063,0.602097
